# 02 — Transport Phenotype (Figure 2)

**Question**: Do football teams exhibit broad, heavy-tailed centroid-transport statistics
consistent with a collective Lévy-type process?

## Panels
| Panel | Content |
|-------|---------|
| **A** | CCDF of centroid run durations $P(T \geq t)$ — log-log, with exp. reference |
| **B** | CCDF of centroid run lengths $P(L \geq \ell)$ — log-log |
| **C** | MSD($\tau$): player, centroid, relative — log-log with power-law fits |
| **D** | MSD decomposition bar chart: fraction of player MSD from centroid component |

**Key result**: Player runs are broad-tailed and substantially driven by centroid motion.

**Prerequisite**: Run `01_data_loading.ipynb` first.

In [ ]:
import sys, os
from pathlib import Path

_HERE = Path(os.getcwd())
_REPO = _HERE.parents[2]
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from analysis.levy_paper.util.paper_utils import (
    configure_paper_plotting,
    load_cache, save_figure,
    plot_ccdf, plot_msd,
    STATE_COLORS,
)
configure_paper_plotting()

# ── Load caches ────────────────────────────────────────────────────────────
runs_long   = load_cache("runs_long")
msd_long    = load_cache("msd_long")

print(f"Runs:  {len(runs_long):,}  |  MSD lags: {msd_long['lag_s'].nunique()}")

## Panel A — CCDF of centroid run durations

A purely exponential process would appear as a straight line on a semi-log plot.
Curvature on a log-log plot (power-law region) indicates heavy tails.

In [ ]:
centroid_runs = runs_long.loc[runs_long["entity"] == "centroid"].copy()
player_runs   = runs_long.loc[runs_long["entity"] == "player"].copy()

print(f"Centroid runs: {len(centroid_runs):,}")
print(f"Player runs:   {len(player_runs):,}")
print(centroid_runs[["duration_s", "length_m", "speed_mps"]].describe())

In [ ]:
fig_a, ax_a = plt.subplots(figsize=(4.5, 3.5))

plot_ccdf(ax_a, centroid_runs["duration_s"],
          label="Centroid", color="#1b7837", reference_exp=True)
plot_ccdf(ax_a, player_runs["duration_s"],
          label="Player",   color="#762a83", ls="--")

ax_a.set_xlabel("Run duration $T$ (s)")
ax_a.set_ylabel(r"$P(T \geq t)$")
ax_a.set_title("A — Run duration CCDF")
ax_a.legend()
plt.tight_layout()
plt.show()

## Panel B — CCDF of centroid run lengths

In [ ]:
fig_b, ax_b = plt.subplots(figsize=(4.5, 3.5))

plot_ccdf(ax_b, centroid_runs["length_m"],
          label="Centroid", color="#1b7837", reference_exp=True)
plot_ccdf(ax_b, player_runs["length_m"],
          label="Player",   color="#762a83", ls="--")

ax_b.set_xlabel("Run length $L$ (m)")
ax_b.set_ylabel(r"$P(L \geq \ell)$")
ax_b.set_title("B — Run length CCDF")
ax_b.legend()
plt.tight_layout()
plt.show()

## Panel C — Mean Squared Displacement

$$\text{MSD}(\tau) = \langle |\mathbf{r}(t+\tau) - \mathbf{r}(t)|^2 \rangle_t$$

Decomposition:
$$\text{MSD}_{\text{player}} = \text{MSD}_{\text{centroid}} + \text{MSD}_{\text{relative}} + \text{cross terms}$$

- $\alpha > 1$: superdiffusive (directed, persistent)
- $\alpha = 1$: normal diffusion
- $\alpha < 1$: subdiffusive

In [ ]:
msd_pivot = msd_long.groupby(["component", "lag_s"])["msd_m2"].mean().reset_index()

fig_c, ax_c = plt.subplots(figsize=(4.5, 3.5))

alphas = {}
palette = {"player": "#762a83", "centroid": "#1b7837", "relative": "#d6604d"}

for component, grp in msd_pivot.groupby("component"):
    tau = grp["lag_s"].values
    msd = grp["msd_m2"].values
    color = palette.get(component, "grey")
    alpha = plot_msd(ax_c, tau, msd,
                     label=component.capitalize(),
                     color=color, fit_alpha=True)
    if alpha is not None:
        alphas[component] = alpha

ax_c.set_title("C — MSD decomposition")
ax_c.legend(fontsize=8)
print("Fitted exponents:", {k: f"{v:.2f}" for k, v in alphas.items()})
plt.tight_layout()
plt.show()

## Panel D — MSD Fraction Explained by Centroid

At each lag $\tau$, what fraction of player MSD is attributable to centroid motion?

In [ ]:
msd_wide = msd_pivot.pivot(index="lag_s", columns="component", values="msd_m2")

# Representative lags (log-spaced)
lags_plot = np.array([1, 5, 10, 30, 60, 120])
lags_plot = lags_plot[np.isin(lags_plot, msd_wide.index)]

fractions = []
for lag in lags_plot:
    row = msd_wide.loc[lag]
    frac = row.get("centroid", np.nan) / row.get("player", np.nan)
    fractions.append(frac)

fig_d, ax_d = plt.subplots(figsize=(4.5, 3.5))
ax_d.bar(range(len(lags_plot)), fractions, color="#1b7837", edgecolor="k", lw=0.5)
ax_d.axhline(1.0, color="k", lw=0.8, ls="--")
ax_d.set_xticks(range(len(lags_plot)))
ax_d.set_xticklabels([f"{l}s" for l in lags_plot])
ax_d.set_ylim(0, 1.1)
ax_d.set_xlabel(r"Lag $\tau$")
ax_d.set_ylabel(r"MSD$_\mathrm{centroid}$ / MSD$_\mathrm{player}$")
ax_d.set_title("D — Centroid fraction of player MSD")
plt.tight_layout()
plt.show()

## Assemble and Save Figure 2

In [ ]:
fig2, axes = plt.subplots(2, 2, figsize=(9, 7))
ax_A, ax_B, ax_C, ax_D = axes.flat

# --- A ---
plot_ccdf(ax_A, centroid_runs["duration_s"], label="Centroid", color="#1b7837", reference_exp=True)
plot_ccdf(ax_A, player_runs["duration_s"],   label="Player",   color="#762a83", ls="--")
ax_A.set_xlabel("Run duration $T$ (s)")
ax_A.set_ylabel(r"$P(T \geq t)$")
ax_A.set_title("A")
ax_A.legend(fontsize=8)

# --- B ---
plot_ccdf(ax_B, centroid_runs["length_m"], label="Centroid", color="#1b7837", reference_exp=True)
plot_ccdf(ax_B, player_runs["length_m"],   label="Player",   color="#762a83", ls="--")
ax_B.set_xlabel("Run length $L$ (m)")
ax_B.set_ylabel(r"$P(L \geq \ell)$")
ax_B.set_title("B")
ax_B.legend(fontsize=8)

# --- C ---
for component, grp in msd_pivot.groupby("component"):
    tau = grp["lag_s"].values
    msd = grp["msd_m2"].values
    plot_msd(ax_C, tau, msd, label=component.capitalize(),
             color=palette.get(component, "grey"), fit_alpha=True)
ax_C.set_title("C")
ax_C.legend(fontsize=7)

# --- D ---
ax_D.bar(range(len(lags_plot)), fractions, color="#1b7837", edgecolor="k", lw=0.5)
ax_D.axhline(1.0, color="k", lw=0.8, ls="--")
ax_D.set_xticks(range(len(lags_plot)))
ax_D.set_xticklabels([f"{l}s" for l in lags_plot])
ax_D.set_ylim(0, 1.1)
ax_D.set_xlabel(r"Lag $\tau$")
ax_D.set_ylabel(r"MSD$_\mathrm{centroid}$ / MSD$_\mathrm{player}$")
ax_D.set_title("D")

fig2.suptitle("Figure 2 — Transport Phenotype", fontsize=13, y=1.01)
save_figure(fig2, "figure2_transport_phenotype")
plt.show()
print("Saved to figures/figure2_transport_phenotype.{pdf,png}")